# Modelado supervisado

### Contexto y objetivo

En las notebooks anteriores exploramos los datos, construimos una tabla integrada por pozo y mes y preparamos variables predictoras a partir del historial de producción, las características de los pozos y la información de fracturación.

En esta notebook entrenaremos y evaluaremos modelos supervisados para predecir la producción de petróleo del mes siguiente. Comenzaremos con regresión y utilizaremos las particiones temporales preparadas en la etapa anterior.

La evaluación se realizará en orden cronológico: usaremos el conjunto de entrenamiento para ajustar los modelos, el de validación para compararlos y el de test para medir el desempeño final. Antes de entrenar, compararemos los modelos con una referencia sencilla basada en la producción del mes de corte.

Una vez evaluada la regresión, abordaremos la clasificación como un problema separado.

### 1. Carga y verificación de los datasets de modelado

La notebook `03_feature_engineering.ipynb` exportó los conjuntos de entrenamiento, validación, test e inferencia, además de un archivo con la definición de las variables. En esta sección cargamos esos archivos y comprobamos sus dimensiones, períodos y columnas antes de comenzar el modelado.

Los conjuntos de entrenamiento, validación y test contienen el valor real de producción del mes siguiente. El conjunto de inferencia representa los registros para los que queremos generar una predicción y, por ese motivo, no tiene ese valor objetivo.

In [2]:
import json
from pathlib import Path

import pandas as pd

# Buscamos la carpeta del proyecto a partir de la ubicación de la notebook.
ubicacion = Path.cwd()
ROOT = next(
    (
        ruta
        for ruta in [ubicacion, *ubicacion.parents]
        if (ruta / "data" / "processed").is_dir()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "No encontramos la carpeta data/processed. "
        "Revisá desde qué carpeta se abrió la notebook."
    )

RUTA_PROCESSED = ROOT / "data" / "processed"

# Estos archivos deben haberse generado al ejecutar la notebook 03.
archivos_requeridos = [
    "train_modelado.csv",
    "validation_modelado.csv",
    "test_modelado.csv",
    "inferencia_modelado.csv",
    "columnas_modelado.json",
]

archivos_faltantes = [
    nombre
    for nombre in archivos_requeridos
    if not (RUTA_PROCESSED / nombre).is_file()
]

if archivos_faltantes:
    raise FileNotFoundError(
        "Faltan archivos de modelado en data/processed: "
        + ", ".join(archivos_faltantes)
        + ". Revisá que las notebooks 02 y 03 hayan terminado de ejecutarse correctamente."
    )

# Cargamos la definición de predictores, objetivos y particiones.
with open(RUTA_PROCESSED / "columnas_modelado.json", encoding="utf-8") as archivo:
    definicion_modelado = json.load(archivo)

# Cargamos las particiones y reconocemos las columnas de fecha.
fechas = ["fecha", "mes_objetivo"]

train = pd.read_csv(
    RUTA_PROCESSED / "train_modelado.csv",
    parse_dates=fechas,
    low_memory=False
)

validation = pd.read_csv(
    RUTA_PROCESSED / "validation_modelado.csv",
    parse_dates=fechas,
    low_memory=False
)

test = pd.read_csv(
    RUTA_PROCESSED / "test_modelado.csv",
    parse_dates=fechas,
    low_memory=False
)

inferencia = pd.read_csv(
    RUTA_PROCESSED / "inferencia_modelado.csv",
    parse_dates=fechas,
    low_memory=False
)

# Resumimos la cantidad de registros, pozos y períodos de cada conjunto.
resumen_particiones = []

for nombre, datos in [
    ("Entrenamiento", train),
    ("Validación", validation),
    ("Test", test),
    ("Inferencia", inferencia),
]:
    resumen_particiones.append({
        "Conjunto": nombre,
        "Filas": len(datos),
        "Columnas": datos.shape[1],
        "Pozos": datos["idpozo"].nunique(),
        "Desde": datos["fecha"].min().strftime("%Y-%m"),
        "Hasta": datos["fecha"].max().strftime("%Y-%m"),
    })

display(pd.DataFrame(resumen_particiones).set_index("Conjunto"))

# Comprobamos que los predictores definidos estén presentes en las particiones.
features = (
    definicion_modelado["features_numericas"]
    + definicion_modelado["features_booleanas"]
    + definicion_modelado["features_categoricas"]
)

columnas_ausentes = sorted(set(features) - set(train.columns))

if columnas_ausentes:
    raise KeyError(
        "Hay predictores definidos en columnas_modelado.json que no aparecen "
        f"en train_modelado.csv: {columnas_ausentes}"
    )

print(f"\nCantidad de variables predictoras definidas: {len(features)}")
print(f"Objetivo de regresión: {definicion_modelado['target_regresion']}")
print(f"Objetivo de clasificación: {definicion_modelado['target_clasificacion']}")
print("\nPeríodos del objetivo por partición:")
for nombre, datos in [
    ("Entrenamiento", train),
    ("Validación", validation),
    ("Test", test),
]:
    print(
        f"- {nombre}: "
        f"{datos['mes_objetivo'].min():%Y-%m} a "
        f"{datos['mes_objetivo'].max():%Y-%m}"
    )

,Filas,Columnas,Pozos,Desde,Hasta
Conjunto,,,,,
Entrenamiento,49452,58,4707,2025-01,2025-11
Validación,14320,58,4814,2025-12,2026-02
Test,19737,58,5015,2026-03,2026-06
Inferencia,5062,56,5062,2026-07,2026-07



Cantidad de variables predictoras definidas: 41
Objetivo de regresión: prod_pet_mes_siguiente
Objetivo de clasificación: clase_prod_pet_mes_siguiente

Períodos del objetivo por partición:
- Entrenamiento: 2025-02 a 2025-12
- Validación: 2026-01 a 2026-03
- Test: 2026-04 a 2026-07


### 2. Modelo de referencia para regresión

Antes de entrenar modelos de Machine Learning, establecemos una referencia sencilla para comparar los resultados. Suponemos que la producción de petróleo del mes siguiente será igual a la producción del mes de corte.

Esta predicción se conoce como persistencia. Por ejemplo, si un pozo produjo 80 m³ durante el mes de corte, la referencia estima que producirá 80 m³ el mes siguiente.

Evaluamos esta referencia con el conjunto de validación, que utilizaremos para comparar modelos. Por el momento no calculamos métricas sobre el conjunto de test, ya que lo reservaremos para la evaluación final.

In [3]:
import numpy as np

# Verificamos que todas las particiones tengan las variables predictoras esperadas.
particiones = {
    "Entrenamiento": train,
    "Validación": validation,
    "Test": test,
    "Inferencia": inferencia,
}

for nombre, datos in particiones.items():
    faltantes = sorted(set(features) - set(datos.columns))

    if faltantes:
        raise KeyError(
            f"Faltan variables predictoras en {nombre}: {faltantes}"
        )

# El objetivo debe estar disponible en entrenamiento, validación y test,
# pero no en el conjunto que se utilizará para generar predicciones futuras.
objetivo = definicion_modelado["target_regresion"]

for nombre, datos in [
    ("Entrenamiento", train),
    ("Validación", validation),
    ("Test", test),
]:
    if objetivo not in datos.columns:
        raise KeyError(f"No se encontró el objetivo de regresión en {nombre}.")

    if datos[objetivo].isna().any():
        raise ValueError(f"Hay valores faltantes en el objetivo de {nombre}.")

if objetivo in inferencia.columns:
    raise ValueError(
        "El conjunto de inferencia no debería incluir el objetivo, "
        "porque todavía no conocemos la producción del mes siguiente."
    )

# En la referencia, usamos la producción del mes de corte
# como estimación de la producción del mes siguiente.
y_real_validacion = validation[objetivo]
y_pred_referencia = validation["prod_pet"]

if y_pred_referencia.isna().any():
    raise ValueError(
        "Hay valores faltantes en prod_pet de validación y no se puede "
        "calcular la referencia para todos los registros."
    )

error = y_real_validacion - y_pred_referencia

mae_referencia = np.abs(error).mean()
rmse_referencia = np.sqrt((error ** 2).mean())

print(f"Registros evaluados en validación: {len(validation):,}")
print(f"MAE de la referencia: {mae_referencia:.2f} m³")
print(f"RMSE de la referencia: {rmse_referencia:.2f} m³")
print(
    "Porcentaje de valores reales de producción iguales a cero: "
    f"{(y_real_validacion == 0).mean() * 100:.1f}%"
)

Registros evaluados en validación: 14,320
MAE de la referencia: 160.04 m³
RMSE de la referencia: 506.07 m³
Porcentaje de valores reales de producción iguales a cero: 30.1%


### 3. Distribución del objetivo y análisis de la referencia

La referencia de persistencia predice que la producción del mes siguiente será igual a la producción del mes de corte. Antes de entrenar modelos, analizamos la distribución de la variable objetivo y los errores de esta referencia.

Este análisis nos permitirá interpretar las métricas generales y observar si la referencia se comporta de manera diferente cuando la producción real del mes siguiente es igual a cero o es positiva. Utilizamos el conjunto de validación para este análisis y mantenemos el conjunto de test apartado para la evaluación final.

In [4]:
# Resumimos la distribución de la producción futura en cada partición.
resumen_objetivo = []

for nombre, datos in [
    ("Entrenamiento", train),
    ("Validación", validation),
    ("Test", test),
]:
    objetivo_datos = datos[objetivo]

    resumen_objetivo.append({
        "Conjunto": nombre,
        "Registros": len(datos),
        "Producción media (m³)": objetivo_datos.mean(),
        "Producción mediana (m³)": objetivo_datos.median(),
        "Percentil 90 (m³)": objetivo_datos.quantile(0.90),
        "Percentil 99 (m³)": objetivo_datos.quantile(0.99),
        "Producción igual a cero (%)": (objetivo_datos == 0).mean() * 100,
    })

resumen_objetivo = pd.DataFrame(resumen_objetivo).set_index("Conjunto")
display(resumen_objetivo.round(2))

# Analizamos por separado los errores de la referencia
# según la producción real del mes siguiente.
analisis_referencia = validation[
    [objetivo, "prod_pet"]
].copy()

analisis_referencia["grupo_produccion_real"] = np.where(
    analisis_referencia[objetivo] == 0,
    "Producción real igual a cero",
    "Producción real positiva"
)

resultados_por_grupo = []

for grupo, datos in analisis_referencia.groupby(
    "grupo_produccion_real",
    observed=True
):
    errores = datos[objetivo] - datos["prod_pet"]

    resultados_por_grupo.append({
        "Grupo": grupo,
        "Registros": len(datos),
        "Porcentaje de validación (%)": len(datos) / len(validation) * 100,
        "MAE de la referencia (m³)": np.abs(errores).mean(),
        "RMSE de la referencia (m³)": np.sqrt((errores ** 2).mean()),
    })

display(pd.DataFrame(resultados_por_grupo).set_index("Grupo").round(2))

,Registros,Producción media (m³),Producción mediana (m³),Percentil 90 (m³),Percentil 99 (m³),Producción igual a cero (%)
Conjunto,,,,,,
Entrenamiento,49452,545.12,12.50,1768.62,6057.91,32.05
Validación,14320,596.48,15.69,1838.86,6743.40,30.13
Test,19737,615.72,18.07,1883.95,6343.12,29.09


,Registros,Porcentaje de validación (%),MAE de la referencia (m³),RMSE de la referencia (m³)
Grupo,,,,
Producción real igual a cero,4315,30.13,11.77,140.66
Producción real positiva,10005,69.87,223.99,598.36


### 4. Preparación de las variables para regresión

Para entrenar un modelo necesitamos separar las variables predictoras del valor que queremos estimar. Utilizaremos las 41 variables definidas en la notebook 03 como entradas (`X`) y `prod_pet_mes_siguiente` como objetivo (`y`).

Cada fila representa un pozo en un mes de corte: las variables de entrada describen la información disponible hasta ese mes y el objetivo contiene la producción del mes calendario siguiente. El objetivo no se incluye entre las variables predictoras.

En esta etapa preparamos las matrices de entrenamiento y validación y revisamos los tipos de datos y los valores faltantes. Todavía no imputamos ni codificamos variables. Esas transformaciones se incorporarán junto con el modelo para que se ajusten únicamente con los datos de entrenamiento. El conjunto de test se mantiene apartado.

In [5]:
# Verificamos que el objetivo no esté incluido entre las variables predictoras.
if objetivo in features:
    raise ValueError(
        f"El objetivo {objetivo} no debe formar parte de las variables predictoras."
    )

# Separamos las entradas (X) y el objetivo (y) para entrenamiento y validación.
X_train = train[features].copy()
y_train = train[objetivo].copy()

X_validation = validation[features].copy()
y_validation = validation[objetivo].copy()

# Revisamos que los objetivos estén completos.
if y_train.isna().any() or y_validation.isna().any():
    raise ValueError("Hay valores faltantes en los objetivos de entrenamiento o validación.")

print(f"Variables predictoras: {len(features)}")
print(f"Filas de entrenamiento: {len(X_train):,}")
print(f"Filas de validación: {len(X_validation):,}")
print(f"Objetivo: {objetivo}")

# Comparamos los tipos de datos y los faltantes de las variables de entrada.
resumen_variables = pd.DataFrame({
    "Tipo de dato": X_train.dtypes.astype(str),
    "% faltantes en entrenamiento": (X_train.isna().mean() * 100).round(2),
    "% faltantes en validación": (X_validation.isna().mean() * 100).round(2),
})

display(
    resumen_variables
    .sort_values("% faltantes en entrenamiento", ascending=False)
)

print("\nCantidad de variables por tipo de dato:")
display(X_train.dtypes.astype(str).value_counts().to_frame("Cantidad"))

Variables predictoras: 41
Filas de entrenamiento: 49,452
Filas de validación: 14,320
Objetivo: prod_pet_mes_siguiente


,Tipo de dato,% faltantes en entrenamiento,% faltantes en validación
prod_pet_cambio_rel,float64,33.13,31.30
potencia_equipos_fractura_hp_max,float64,29.66,26.56
presion_maxima_psi_max,float64,29.37,26.28
longitud_rama_horizontal_m_max,float64,29.35,26.26
arena_bombeada_tn_acum,float64,29.35,26.26
agua_inyectada_m3_acum,float64,29.35,26.26
cantidad_fracturas_acum,float64,29.35,26.26
meses_desde_ultima_fractura,float64,29.35,26.26
eventos_fractura_acum,float64,29.35,26.26
tipo_terminacion_ultima,str,29.35,26.26



Cantidad de variables por tipo de dato:


,Cantidad
float64,26
str,11
int64,2
bool,2


### 5. Primer modelo de regresión

En esta sección entrenamos un primer modelo para estimar `prod_pet_mes_siguiente`. Utilizamos como variables predictoras las columnas definidas en la notebook 03 y ajustamos el modelo únicamente con el conjunto de entrenamiento.

Antes del entrenamiento, incorporamos dentro de un `Pipeline` el tratamiento de valores faltantes y la codificación de las variables categóricas. De esta manera, las transformaciones se ajustan con los datos de entrenamiento y se aplican sin volver a estimarlas sobre validación.

Como primer algoritmo utilizamos `RandomForestRegressor`. Este modelo combina varios árboles de decisión y permite representar relaciones no lineales entre las variables. En esta etapa no buscamos encontrar la configuración óptima: queremos obtener una primera medida de desempeño y compararla con la referencia de persistencia.

Evaluamos el resultado con MAE, RMSE y R² sobre validación. El conjunto de test se mantiene apartado para la evaluación final.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Recuperamos los grupos de variables según su tipo.
features_numericas = definicion_modelado["features_numericas"]
features_booleanas = definicion_modelado["features_booleanas"]
features_categoricas = definicion_modelado["features_categoricas"]

# Las variables booleanas se pueden tratar como entradas numéricas.
columnas_numericas = features_numericas + features_booleanas

# Para los faltantes numéricos usamos medianas aprendidas solo en entrenamiento.
preprocesamiento_numerico = Pipeline([
    ("imputador", SimpleImputer(strategy="median", add_indicator=True))
])

# Los faltantes categóricos se identifican explícitamente.
# OneHotEncoder convierte las categorías en columnas numéricas.
preprocesamiento_categorico = Pipeline([
    ("imputador", SimpleImputer(strategy="constant", fill_value="Sin dato")),
    ("codificador", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=20
    ))
])

# Aplicamos un preprocesamiento distinto según el tipo de variable.
preprocesador = ColumnTransformer(
    transformers=[
        ("numericas", preprocesamiento_numerico, columnas_numericas),
        ("categoricas", preprocesamiento_categorico, features_categoricas),
    ],
    remainder="drop"
)

# Creamos un primer modelo de bosque aleatorio.
modelo_regresion = Pipeline([
    ("preprocesamiento", preprocesador),
    ("modelo", RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

# Ajustamos el preprocesamiento y el modelo usando solamente entrenamiento.
modelo_regresion.fit(X_train, y_train)

# Generamos predicciones para validación, que no se usó para ajustar el modelo.
y_pred_modelo = modelo_regresion.predict(X_validation)

# Calculamos las métricas de la referencia y del primer modelo.
resultados_modelos = pd.DataFrame({
    "MAE (m³)": [
        mean_absolute_error(y_validation, y_pred_referencia),
        mean_absolute_error(y_validation, y_pred_modelo),
    ],
    "RMSE (m³)": [
        np.sqrt(mean_squared_error(y_validation, y_pred_referencia)),
        np.sqrt(mean_squared_error(y_validation, y_pred_modelo)),
    ],
    "R²": [
        r2_score(y_validation, y_pred_referencia),
        r2_score(y_validation, y_pred_modelo),
    ],
}, index=["Persistencia", "Random Forest"])

display(resultados_modelos.round(3))

,MAE (m³),RMSE (m³),R²
Persistencia,160.039,506.071,0.85
Random Forest,128.398,368.437,0.92


### 6. Análisis de errores en validación

Las métricas generales resumen el desempeño de cada método, pero no muestran en qué tipo de registros se producen los errores. Por eso comparamos la referencia de persistencia y el Random Forest en dos grupos: observaciones cuya producción real del mes siguiente fue igual a cero y observaciones con producción positiva.

Para cada grupo calculamos el MAE y el RMSE. También calculamos el error promedio con signo, definido como predicción menos valor real: un resultado positivo indica que, en promedio, se sobreestima la producción; uno negativo indica que se la subestima.

Este análisis se realiza en validación. El conjunto de test continúa apartado y no se utiliza para elegir ni ajustar modelos.

In [7]:
# Reunimos los valores reales y las predicciones de validación.
analisis_errores = pd.DataFrame({
    "Producción real": y_validation,
    "Persistencia": y_pred_referencia,
    "Random Forest": y_pred_modelo,
})

# Separamos los registros según la producción real del mes siguiente.
analisis_errores["Grupo"] = np.where(
    analisis_errores["Producción real"] == 0,
    "Producción real igual a cero",
    "Producción real positiva"
)

resultados_errores = []

for grupo, datos in analisis_errores.groupby("Grupo", sort=False):
    y_real_grupo = datos["Producción real"]

    for nombre_modelo in ["Persistencia", "Random Forest"]:
        y_pred_grupo = datos[nombre_modelo]
        error = y_pred_grupo - y_real_grupo

        resultados_errores.append({
            "Grupo": grupo,
            "Modelo": nombre_modelo,
            "Registros": len(datos),
            "MAE (m³)": np.abs(error).mean(),
            "RMSE (m³)": np.sqrt((error ** 2).mean()),
            "Error promedio (m³)": error.mean(),
            "Producción real media (m³)": y_real_grupo.mean(),
            "Predicción media (m³)": y_pred_grupo.mean(),
        })

resultados_errores = (
    pd.DataFrame(resultados_errores)
    .set_index(["Grupo", "Modelo"])
)

display(resultados_errores.round(2))

Registros  MAE (m³)  RMSE (m³)  \
Grupo                        Modelo                                          
Producción real positiva     Persistencia       10005    223.99     598.36   
                             Random Forest      10005    174.43     428.03   
Producción real igual a cero Persistencia        4315     11.77     140.66   
                             Random Forest       4315     21.66     160.31   

                                            Error promedio (m³)  \
Grupo                        Modelo                               
Producción real positiva     Persistencia                 -3.93   
                             Random Forest               -14.74   
Producción real igual a cero Persistencia                 11.77   
                             Random Forest                21.66   

                                            Producción real media (m³)  \
Grupo                        Modelo                                      
Producción real positiva     Persistencia                       853.73   
                             Random Forest                      853.73   
Producción real igual a cero Persistencia                         0.00   
                             Random Forest                        0.00   

                                            Predicción media (m³)  
Grupo                        Modelo                                
Producción real positiva     Persistencia                  849.81  
                             Random Forest                 838.99  
Producción real igual a cero Persistencia                   11.77  
                             Random Forest                  21.66

### 7. Comparación con un modelo de regresión lineal

El análisis de errores muestra que Random Forest mejora a la referencia de persistencia en los registros con producción positiva, pero presenta errores mayores en los registros cuya producción real es igual a cero. Para ampliar la comparación, entrenamos un modelo Ridge, que es una regresión lineal regularizada.

Ridge estima la producción a partir de una combinación lineal de las variables predictoras y aplica una penalización para limitar coeficientes demasiado grandes. A diferencia de Random Forest, no construye árboles ni modela automáticamente relaciones no lineales. Por ese motivo, esta comparación nos permitirá evaluar si un modelo más simple puede obtener resultados competitivos.

Las variables numéricas se imputan y escalan, y las categóricas se imputan y codifican. Estas transformaciones se ajustan únicamente con entrenamiento y se aplican a validación. Comparamos ambos modelos y la referencia con las mismas métricas y particiones. El conjunto de test continúa apartado.

In [8]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Preparamos las variables numéricas para Ridge:
# completamos faltantes y escalamos las variables.
preprocesamiento_numerico_ridge = Pipeline([
    ("imputador", SimpleImputer(strategy="median", add_indicator=True)),
    ("escalador", StandardScaler())
])

# Completamos faltantes categóricos y codificamos las categorías.
preprocesamiento_categorico_ridge = Pipeline([
    ("imputador", SimpleImputer(strategy="constant", fill_value="Sin dato")),
    ("codificador", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=20
    ))
])

# Aplicamos el tratamiento correspondiente a cada grupo de variables.
preprocesador_ridge = ColumnTransformer(
    transformers=[
        ("numericas", preprocesamiento_numerico_ridge, columnas_numericas),
        ("categoricas", preprocesamiento_categorico_ridge, features_categoricas),
    ],
    remainder="drop"
)

# Creamos una primera versión de Ridge.
# alpha controla la intensidad de la regularización.
modelo_ridge = Pipeline([
    ("preprocesamiento", preprocesador_ridge),
    ("modelo", Ridge(alpha=10.0))
])

# Ajustamos el preprocesamiento y el modelo solo con los datos de entrenamiento.
modelo_ridge.fit(X_train, y_train)

# Predecimos la producción del mes siguiente para validación.
y_pred_ridge = modelo_ridge.predict(X_validation)

# Comparamos los tres métodos sobre las mismas observaciones de validación.
predicciones_validacion = {
    "Persistencia": y_pred_referencia,
    "Random Forest": y_pred_modelo,
    "Ridge": y_pred_ridge,
}

comparacion_validacion = pd.DataFrame(
    {
        "MAE (m³)": {
            nombre: mean_absolute_error(y_validation, prediccion)
            for nombre, prediccion in predicciones_validacion.items()
        },
        "RMSE (m³)": {
            nombre: np.sqrt(mean_squared_error(y_validation, prediccion))
            for nombre, prediccion in predicciones_validacion.items()
        },
        "R²": {
            nombre: r2_score(y_validation, prediccion)
            for nombre, prediccion in predicciones_validacion.items()
        },
    }
)

display(comparacion_validacion.round(3))

# Revisamos si Ridge produce estimaciones negativas.
print(
    "Predicciones negativas de Ridge en validación: "
    f"{(y_pred_ridge < 0).sum():,} "
    f"({(y_pred_ridge < 0).mean() * 100:.2f}%)"
)

,MAE (m³),RMSE (m³),R²
Persistencia,160.039,506.071,0.850
Random Forest,128.398,368.437,0.920
Ridge,166.814,402.679,0.905


Predicciones negativas de Ridge en validación: 4,190 (29.26%)


### 8. Resultados iniciales y revisión de predicciones negativas

En las primeras comparaciones, Random Forest obtuvo menores valores de MAE y RMSE que la referencia de persistencia en el conjunto de validación. Al analizar los errores por separado, observamos que mejora las predicciones cuando la producción real es positiva, pero presenta errores mayores cuando la producción real es igual a cero.

Ridge obtuvo un MAE algo mayor que la referencia y generó predicciones negativas, aunque la producción de petróleo no puede ser negativa. Para evaluar el efecto de esta limitación, calculamos una variante en la que reemplazamos por cero las predicciones negativas de Ridge. Comparamos sus métricas con las de los métodos anteriores, usando las mismas observaciones de validación.

Esta corrección se evalúa como un experimento explícito: no cambia los valores reales ni el entrenamiento de Ridge. El conjunto de test se mantiene apartado.

In [9]:
# Aplicamos una restricción física a las predicciones de Ridge:
# la producción estimada no puede ser menor que cero.
y_pred_ridge_no_negativa = np.maximum(y_pred_ridge, 0)

# Comparamos los métodos sobre las mismas observaciones de validación.
predicciones_comparacion = {
    "Persistencia": y_pred_referencia,
    "Random Forest": y_pred_modelo,
    "Ridge": y_pred_ridge,
    "Ridge con mínimo cero": y_pred_ridge_no_negativa,
}

resultados_comparacion = pd.DataFrame(
    {
        "MAE (m³)": {
            nombre: mean_absolute_error(y_validation, prediccion)
            for nombre, prediccion in predicciones_comparacion.items()
        },
        "RMSE (m³)": {
            nombre: np.sqrt(mean_squared_error(y_validation, prediccion))
            for nombre, prediccion in predicciones_comparacion.items()
        },
        "R²": {
            nombre: r2_score(y_validation, prediccion)
            for nombre, prediccion in predicciones_comparacion.items()
        },
        "Predicciones negativas": {
            nombre: (prediccion < 0).sum()
            for nombre, prediccion in predicciones_comparacion.items()
        },
    }
)

display(resultados_comparacion.round(3))

,MAE (m³),RMSE (m³),R²,Predicciones negativas
Persistencia,160.039,506.071,0.850,0
Random Forest,128.398,368.437,0.920,0
Ridge,166.814,402.679,0.905,4190
Ridge con mínimo cero,154.138,401.138,0.905,0


### 9. Validación temporal del primer modelo

En la comparación inicial, Random Forest obtuvo mejores métricas que la referencia de persistencia y Ridge en el conjunto de validación. Para comprobar si su desempeño se mantiene en distintos períodos, realizamos una validación temporal dentro del conjunto de entrenamiento.

En cada pliegue, entrenamos con los meses del objetivo anteriores y evaluamos en uno o más meses posteriores. No mezclamos los períodos de manera aleatoria, ya que queremos simular el uso del modelo para predecir el futuro.

Comparamos Random Forest con la referencia de persistencia utilizando MAE y RMSE. Esta validación sirve para estudiar la estabilidad del modelo y no reemplaza la evaluación final en test, que continúa apartado.

In [10]:
from sklearn.base import clone

# Definimos pliegues cronológicos usando el mes del objetivo.
# En cada pliegue, los meses de validación son posteriores a los de entrenamiento.
pliegues_temporales = [
    {
        "Pliegue": "1",
        "Entrenamiento hasta": "2025-07-01",
        "Validación desde": "2025-08-01",
        "Validación hasta": "2025-09-01",
    },
    {
        "Pliegue": "2",
        "Entrenamiento hasta": "2025-09-01",
        "Validación desde": "2025-10-01",
        "Validación hasta": "2025-11-01",
    },
    {
        "Pliegue": "3",
        "Entrenamiento hasta": "2025-11-01",
        "Validación desde": "2025-12-01",
        "Validación hasta": "2025-12-01",
    },
]

resultados_pliegues = []
objetivos_reales_pliegues = []
predicciones_rf_pliegues = []
predicciones_persistencia_pliegues = []

for pliegue in pliegues_temporales:
    fin_entrenamiento = pd.Timestamp(pliegue["Entrenamiento hasta"])
    inicio_validacion = pd.Timestamp(pliegue["Validación desde"])
    fin_validacion = pd.Timestamp(pliegue["Validación hasta"])

    mascara_entrenamiento = train["mes_objetivo"] <= fin_entrenamiento
    mascara_validacion = train["mes_objetivo"].between(
        inicio_validacion,
        fin_validacion
    )

    datos_entrenamiento = train.loc[mascara_entrenamiento]
    datos_validacion = train.loc[mascara_validacion]

    if datos_entrenamiento.empty or datos_validacion.empty:
        raise ValueError(
            f"El pliegue {pliegue['Pliegue']} no tiene filas suficientes."
        )

    # Confirmamos que validemos siempre en meses posteriores al entrenamiento.
    if datos_entrenamiento["mes_objetivo"].max() >= datos_validacion["mes_objetivo"].min():
        raise ValueError(
            f"El pliegue {pliegue['Pliegue']} no respeta el orden temporal."
        )

    # Creamos una copia sin ajustar del pipeline y la entrenamos solo en este pliegue.
    modelo_pliegue = clone(modelo_regresion)
    modelo_pliegue.fit(
        datos_entrenamiento[features],
        datos_entrenamiento[objetivo]
    )

    prediccion_rf = modelo_pliegue.predict(datos_validacion[features])
    prediccion_persistencia = datos_validacion["prod_pet"].to_numpy()
    y_real = datos_validacion[objetivo].to_numpy()

    resultados_pliegues.append({
        "Pliegue": pliegue["Pliegue"],
        "Objetivos de validación": (
            f"{datos_validacion['mes_objetivo'].min():%Y-%m} a "
            f"{datos_validacion['mes_objetivo'].max():%Y-%m}"
        ),
        "Filas de entrenamiento": len(datos_entrenamiento),
        "Filas de validación": len(datos_validacion),
        "MAE persistencia (m³)": mean_absolute_error(y_real, prediccion_persistencia),
        "MAE Random Forest (m³)": mean_absolute_error(y_real, prediccion_rf),
        "RMSE persistencia (m³)": np.sqrt(
            mean_squared_error(y_real, prediccion_persistencia)
        ),
        "RMSE Random Forest (m³)": np.sqrt(
            mean_squared_error(y_real, prediccion_rf)
        ),
    })

    objetivos_reales_pliegues.extend(y_real)
    predicciones_rf_pliegues.extend(prediccion_rf)
    predicciones_persistencia_pliegues.extend(prediccion_persistencia)

display(pd.DataFrame(resultados_pliegues).set_index("Pliegue").round(2))

# Calculamos también las métricas agrupadas de todas las predicciones temporales.
y_real_pliegues = np.asarray(objetivos_reales_pliegues)
y_rf_pliegues = np.asarray(predicciones_rf_pliegues)
y_persistencia_pliegues = np.asarray(predicciones_persistencia_pliegues)

resumen_validacion_temporal = pd.DataFrame(
    {
        "MAE (m³)": [
            mean_absolute_error(y_real_pliegues, y_persistencia_pliegues),
            mean_absolute_error(y_real_pliegues, y_rf_pliegues),
        ],
        "RMSE (m³)": [
            np.sqrt(mean_squared_error(y_real_pliegues, y_persistencia_pliegues)),
            np.sqrt(mean_squared_error(y_real_pliegues, y_rf_pliegues)),
        ],
    },
    index=["Persistencia", "Random Forest"]
)

print("\nMétricas agrupadas de los pliegues temporales:")
display(resumen_validacion_temporal.round(2))

,Objetivos de validación,Filas de entrenamiento,Filas de validación,MAE persistencia (m³),MAE Random Forest (m³),RMSE persistencia (m³),RMSE Random Forest (m³)
Pliegue,,,,,,,
1,2025-08 a 2025-09,26323,9118,161.31,132.30,522.44,398.26
2,2025-10 a 2025-11,35441,9307,166.55,128.44,559.03,388.73
3,2025-12 a 2025-12,44748,4704,163.76,135.93,562.44,411.44



Métricas agrupadas de los pliegues temporales:


,MAE (m³),RMSE (m³)
Persistencia,163.92,545.62
Random Forest,131.49,397.19


### 10. Ajuste inicial de Random Forest con validación temporal

En los tres pliegues temporales, Random Forest obtuvo menores valores de MAE y RMSE que la referencia de persistencia. Esta consistencia es una señal favorable, aunque los pliegues corresponden a meses de 2025 y no garantizan que el desempeño se mantenga en todos los períodos futuros.

A continuación comparamos un conjunto reducido de configuraciones de Random Forest. Variamos el tamaño mínimo de las hojas y la cantidad de variables consideradas en cada división del árbol. Estos parámetros controlan la complejidad y diversidad del bosque.

Evaluamos todas las configuraciones en los mismos pliegues temporales y con el mismo preprocesamiento. Calculamos las métricas sobre las predicciones de los períodos posteriores al entrenamiento de cada pliegue. Usamos esta comparación para elegir una configuración preliminar; luego la contrastaremos en validación. El conjunto de test continúa apartado.

In [11]:
from sklearn.base import clone

# Comparamos el Random Forest inicial con dos variantes moderadas.
configuraciones_rf = {
    "RF inicial": {
        "n_estimators": 200,
        "min_samples_leaf": 5,
        "max_features": 1.0,
    },
    "RF hojas más grandes": {
        "n_estimators": 200,
        "min_samples_leaf": 10,
        "max_features": 1.0,
    },
    "RF con menos variables por división": {
        "n_estimators": 200,
        "min_samples_leaf": 5,
        "max_features": 0.7,
    },
}

resultados_ajuste_rf = []

for nombre_configuracion, parametros in configuraciones_rf.items():
    predicciones_reales = []
    predicciones_modelo = []
    resultados_por_pliegue = []

    for pliegue in pliegues_temporales:
        fin_entrenamiento = pd.Timestamp(pliegue["Entrenamiento hasta"])
        inicio_validacion = pd.Timestamp(pliegue["Validación desde"])
        fin_validacion = pd.Timestamp(pliegue["Validación hasta"])

        mascara_entrenamiento = train["mes_objetivo"] <= fin_entrenamiento
        mascara_validacion = train["mes_objetivo"].between(
            inicio_validacion,
            fin_validacion
        )

        datos_entrenamiento = train.loc[mascara_entrenamiento]
        datos_validacion = train.loc[mascara_validacion]

        # Partimos del pipeline anterior para mantener el mismo preprocesamiento.
        modelo_pliegue = clone(modelo_regresion)
        modelo_pliegue.set_params(
            modelo__n_estimators=parametros["n_estimators"],
            modelo__min_samples_leaf=parametros["min_samples_leaf"],
            modelo__max_features=parametros["max_features"],
        )

        # Ajustamos imputación, codificación y modelo solo con este pliegue.
        modelo_pliegue.fit(
            datos_entrenamiento[features],
            datos_entrenamiento[objetivo]
        )

        y_real_pliegue = datos_validacion[objetivo].to_numpy()
        y_pred_pliegue = modelo_pliegue.predict(datos_validacion[features])

        predicciones_reales.extend(y_real_pliegue)
        predicciones_modelo.extend(y_pred_pliegue)

        resultados_por_pliegue.append({
            "MAE": mean_absolute_error(y_real_pliegue, y_pred_pliegue),
            "RMSE": np.sqrt(mean_squared_error(y_real_pliegue, y_pred_pliegue)),
        })

    # Calculamos métricas agrupadas sobre todas las predicciones temporales.
    y_real_total = np.asarray(predicciones_reales)
    y_pred_total = np.asarray(predicciones_modelo)

    resultados_ajuste_rf.append({
        "Configuración": nombre_configuracion,
        "min_samples_leaf": parametros["min_samples_leaf"],
        "max_features": parametros["max_features"],
        "MAE agrupado (m³)": mean_absolute_error(y_real_total, y_pred_total),
        "RMSE agrupado (m³)": np.sqrt(
            mean_squared_error(y_real_total, y_pred_total)
        ),
        "MAE medio entre pliegues (m³)": np.mean(
            [resultado["MAE"] for resultado in resultados_por_pliegue]
        ),
        "RMSE medio entre pliegues (m³)": np.mean(
            [resultado["RMSE"] for resultado in resultados_por_pliegue]
        ),
    })

tabla_ajuste_rf = (
    pd.DataFrame(resultados_ajuste_rf)
    .set_index("Configuración")
    .sort_values("MAE agrupado (m³)")
)

display(tabla_ajuste_rf.round(2))

,min_samples_leaf,max_features,MAE agrupado (m³),RMSE agrupado (m³),MAE medio entre pliegues (m³),RMSE medio entre pliegues (m³)
Configuración,,,,,,
RF con menos variables por división,5,0.7,130.51,393.16,131.24,395.27
RF inicial,5,1.0,131.49,397.19,132.22,399.48
RF hojas más grandes,10,1.0,133.01,402.36,133.91,405.05


En esta comparación evaluamos tres configuraciones de Random Forest mediante los mismos pliegues temporales. Variamos `min_samples_leaf`, que controla la cantidad mínima de observaciones por hoja, y `max_features`, que determina qué proporción de variables se considera en cada división. Mantuvimos constantes los demás parámetros y el preprocesamiento.

La configuración con `min_samples_leaf=5` y `max_features=0.7` obtuvo los menores valores de MAE y RMSE. Sin embargo, la mejora respecto del Random Forest inicial fue pequeña. Por este motivo, la consideramos una configuración candidata y no un modelo óptimo demostrado. La comparación se realizó únicamente con pliegues de entrenamiento; el conjunto de test sigue reservado.

### 11. Evaluación de la configuración candidata en validación

En la validación temporal realizada dentro del conjunto de entrenamiento, la configuración de Random Forest con `max_features=0.7` obtuvo los menores valores de MAE y RMSE. La mejora respecto del Random Forest inicial fue pequeña, por lo que la consideramos una configuración candidata y no una mejora confirmada.

En esta sección ajustamos esa variante con todo el conjunto de entrenamiento y la evaluamos en validación, que contiene meses posteriores. Comparamos sus resultados con la referencia de persistencia y con el Random Forest inicial, utilizando las mismas observaciones y métricas.

Usamos esta comparación para decidir qué configuración llevar a la evaluación final. El conjunto de test se mantiene apartado y no se utiliza para ajustar el modelo.

In [12]:
from sklearn.base import clone

# Creamos la variante candidata a partir del pipeline inicial.
# Solo cambiamos max_features; el resto del preprocesamiento y del modelo se mantiene.
modelo_rf_candidato = clone(modelo_regresion)
modelo_rf_candidato.set_params(modelo__max_features=0.7)

# Ajustamos el pipeline completo usando únicamente el conjunto de entrenamiento.
modelo_rf_candidato.fit(X_train, y_train)

# Generamos predicciones para los meses posteriores de validación.
y_pred_rf_candidato = modelo_rf_candidato.predict(X_validation)

# Comparamos todos los métodos sobre las mismas filas de validación.
predicciones_validacion = {
    "Persistencia": y_pred_referencia,
    "Random Forest inicial": y_pred_modelo,
    "Random Forest candidato": y_pred_rf_candidato,
}

comparacion_candidata = pd.DataFrame(
    {
        "MAE (m³)": {
            nombre: mean_absolute_error(y_validation, prediccion)
            for nombre, prediccion in predicciones_validacion.items()
        },
        "RMSE (m³)": {
            nombre: np.sqrt(mean_squared_error(y_validation, prediccion))
            for nombre, prediccion in predicciones_validacion.items()
        },
        "R²": {
            nombre: r2_score(y_validation, prediccion)
            for nombre, prediccion in predicciones_validacion.items()
        },
        "Predicciones negativas": {
            nombre: (prediccion < 0).sum()
            for nombre, prediccion in predicciones_validacion.items()
        },
    }
)

display(comparacion_candidata.round(3))

,MAE (m³),RMSE (m³),R²,Predicciones negativas
Persistencia,160.039,506.071,0.850,0
Random Forest inicial,128.398,368.437,0.920,0
Random Forest candidato,127.128,364.360,0.922,0


En el conjunto de validación, el Random Forest candidato (`max_features=0.7`) obtuvo menores valores de MAE y RMSE que la persistencia y que el Random Forest inicial. La diferencia frente al Random Forest inicial fue pequeña, mientras que la mejora frente a la persistencia fue más marcada. El candidato tampoco produjo estimaciones negativas.

A partir de esta comparación seleccionamos el Random Forest candidato para la evaluación final. Mantenemos sus parámetros sin cambios y no utilizaremos el conjunto de test para elegir configuraciones. En la siguiente sección evaluaremos el modelo una sola vez en los meses posteriores reservados para test y compararemos sus resultados con la persistencia.

### 12. Evaluación final en el conjunto de test

Una vez seleccionado el Random Forest candidato con los resultados de validación, medimos su desempeño en el conjunto de test. Este conjunto contiene meses posteriores a los usados para entrenar y validar, por lo que permite evaluar cómo se comporta el modelo ante períodos futuros.

Comparamos el modelo con la referencia de persistencia usando MAE, RMSE y R². También revisamos el MAE y el RMSE por separado según la producción real del mes siguiente sea cero o positiva, para observar si el desempeño general oculta diferencias entre ambos grupos.

El conjunto de test se utiliza únicamente para esta evaluación final. No modificaremos el modelo ni sus parámetros en función de estos resultados.

In [13]:
# Preparamos la producción real del conjunto de test.
y_test = test[objetivo]

if y_test.isna().any():
    raise ValueError("Hay valores faltantes en el objetivo del conjunto de test.")

# La persistencia predice que el próximo mes tendrá la misma producción
# que el mes de corte.
y_pred_persistencia_test = test["prod_pet"].to_numpy()

# El Random Forest candidato ya fue ajustado con entrenamiento.
# Lo usamos para predecir test sin volver a entrenarlo ni ajustarlo.
X_test = test[features]
y_pred_rf_test = modelo_rf_candidato.predict(X_test)

# Métricas generales de test.
resultados_test = pd.DataFrame(
    {
        "MAE (m³)": [
            mean_absolute_error(y_test, y_pred_persistencia_test),
            mean_absolute_error(y_test, y_pred_rf_test),
        ],
        "RMSE (m³)": [
            np.sqrt(mean_squared_error(y_test, y_pred_persistencia_test)),
            np.sqrt(mean_squared_error(y_test, y_pred_rf_test)),
        ],
        "R²": [
            r2_score(y_test, y_pred_persistencia_test),
            r2_score(y_test, y_pred_rf_test),
        ],
        "Predicciones negativas": [
            (y_pred_persistencia_test < 0).sum(),
            (y_pred_rf_test < 0).sum(),
        ],
    },
    index=["Persistencia", "Random Forest candidato"],
)

display(resultados_test.round(3))

# Analizamos además los errores del modelo elegido por grupo de producción real.
analisis_test = pd.DataFrame({
    "Producción real": y_test.to_numpy(),
    "Predicción Random Forest": y_pred_rf_test,
})

analisis_test["Grupo"] = np.where(
    analisis_test["Producción real"] == 0,
    "Producción real igual a cero",
    "Producción real positiva"
)

resultados_test_por_grupo = []

for grupo, datos in analisis_test.groupby("Grupo", sort=False):
    errores = datos["Predicción Random Forest"] - datos["Producción real"]

    resultados_test_por_grupo.append({
        "Grupo": grupo,
        "Registros": len(datos),
        "Porcentaje de test (%)": len(datos) / len(analisis_test) * 100,
        "MAE (m³)": np.abs(errores).mean(),
        "RMSE (m³)": np.sqrt((errores ** 2).mean()),
        "Error promedio (m³)": errores.mean(),
        "Predicción media (m³)": datos["Predicción Random Forest"].mean(),
    })

display(
    pd.DataFrame(resultados_test_por_grupo)
    .set_index("Grupo")
    .round(2)
)

,MAE (m³),RMSE (m³),R²,Predicciones negativas
Persistencia,166.195,551.710,0.820,0
Random Forest candidato,137.185,409.039,0.901,0


,Registros,Porcentaje de test (%),MAE (m³),RMSE (m³),Error promedio (m³),Predicción media (m³)
Grupo,,,,,,
Producción real positiva,13996,70.91,184.44,476.31,-8.50,859.79
Producción real igual a cero,5741,29.09,21.97,148.68,21.97,21.97


#### Resultados de la evaluación final

En el conjunto de test, el Random Forest candidato obtuvo un MAE de 137,19 m³ y un RMSE de 409,04 m³, frente a 166,20 m³ y 551,71 m³ de la referencia de persistencia. En este período, el modelo redujo el MAE aproximadamente un 17 % y el RMSE aproximadamente un 26 % respecto de la referencia.

El análisis por grupo muestra que el error es distinto según la producción real del mes siguiente. En los registros con producción positiva, el MAE fue de 184,44 m³. En los registros con producción igual a cero, el modelo predijo en promedio 21,97 m³, por lo que tendió a sobreestimar esos casos.

Los resultados de test respaldan el uso del Random Forest candidato frente a la referencia para los meses evaluados, aunque sus errores siguen siendo relevantes y su desempeño es menos preciso en los casos con producción real igual a cero. Consideramos realizada la evaluación final y no utilizaremos test para ajustar nuevos parámetros.

### 13. Diagnóstico del modelo por período y tipo de pozo

El Random Forest candidato mejoró las métricas generales respecto de la referencia, pero el resultado promedio no muestra si el desempeño se mantiene en todos los meses y tipos de pozo.

En esta sección analizamos las predicciones de validación por mes del objetivo y por tipo de pozo. Para cada grupo calculamos MAE y RMSE del Random Forest candidato y de la referencia de persistencia. Este análisis es descriptivo: nos permitirá identificar grupos donde los errores son mayores y orientar los próximos experimentos, sin utilizar el conjunto de test para seleccionar modelos.

In [14]:
# Reunimos los valores reales y las predicciones ya calculadas para validación.
diagnostico_validacion = validation[
    ["mes_objetivo", "tipopozo", objetivo, "prod_pet"]
].copy()

diagnostico_validacion["Predicción persistencia"] = (
    y_pred_referencia
)
diagnostico_validacion["Predicción Random Forest"] = (
    y_pred_rf_candidato
)

diagnostico_validacion["Tipo de pozo"] = (
    diagnostico_validacion["tipopozo"]
    .fillna("Sin dato")
    .astype(str)
)

# Resumimos el desempeño por mes del objetivo.
resumen_por_mes = []

for mes, datos in diagnostico_validacion.groupby("mes_objetivo", sort=True):
    y_real = datos[objetivo]

    resumen_por_mes.append({
        "Mes objetivo": mes.strftime("%Y-%m"),
        "Registros": len(datos),
        "Producción real igual a cero (%)": (y_real == 0).mean() * 100,
        "MAE persistencia (m³)": mean_absolute_error(
            y_real, datos["Predicción persistencia"]
        ),
        "MAE Random Forest (m³)": mean_absolute_error(
            y_real, datos["Predicción Random Forest"]
        ),
        "RMSE persistencia (m³)": np.sqrt(mean_squared_error(
            y_real, datos["Predicción persistencia"]
        )),
        "RMSE Random Forest (m³)": np.sqrt(mean_squared_error(
            y_real, datos["Predicción Random Forest"]
        )),
    })

print("Desempeño por mes objetivo en validación:")
display(pd.DataFrame(resumen_por_mes).set_index("Mes objetivo").round(2))

# Resumimos el desempeño por tipo de pozo.
resumen_por_tipo = []

for tipo, datos in diagnostico_validacion.groupby("Tipo de pozo", sort=True):
    y_real = datos[objetivo]

    resumen_por_tipo.append({
        "Tipo de pozo": tipo,
        "Registros": len(datos),
        "Producción real igual a cero (%)": (y_real == 0).mean() * 100,
        "MAE persistencia (m³)": mean_absolute_error(
            y_real, datos["Predicción persistencia"]
        ),
        "MAE Random Forest (m³)": mean_absolute_error(
            y_real, datos["Predicción Random Forest"]
        ),
        "RMSE persistencia (m³)": np.sqrt(mean_squared_error(
            y_real, datos["Predicción persistencia"]
        )),
        "RMSE Random Forest (m³)": np.sqrt(mean_squared_error(
            y_real, datos["Predicción Random Forest"]
        )),
    })

print("\nDesempeño por tipo de pozo en validación:")
display(
    pd.DataFrame(resumen_por_tipo)
    .set_index("Tipo de pozo")
    .sort_values("Registros", ascending=False)
    .round(2)
)

Desempeño por mes objetivo en validación:


,Registros,Producción real igual a cero (%),MAE persistencia (m³),MAE Random Forest (m³),RMSE persistencia (m³),RMSE Random Forest (m³)
Mes objetivo,,,,,,
2026-01,4736,30.26,148.59,123.02,501.03,373.57
2026-02,4772,30.30,175.57,119.10,500.81,337.37
2026-03,4812,29.84,155.90,139.13,516.10,380.59



Desempeño por tipo de pozo en validación:


,Registros,Producción real igual a cero (%),MAE persistencia (m³),MAE Random Forest (m³),RMSE persistencia (m³),RMSE Random Forest (m³)
Tipo de pozo,,,,,,
Petrolífero,6760,9.59,301.40,233.63,703.47,498.27
Gasífero,6554,40.89,38.43,34.63,221.41,178.77
Otro tipo,922,100.00,0.00,1.96,0.00,13.19
Sumidero,62,100.00,0.00,41.08,0.00,128.44
Inyección de Gas,14,21.43,4.11,4.51,7.93,8.59
Sin dato,8,0.00,294.06,1213.85,317.76,1230.55


En validación, el Random Forest candidato obtuvo menores valores de MAE y RMSE que la persistencia en los tres meses objetivo analizados. La proporción de registros con producción real igual a cero se mantuvo cercana al 30 % en cada mes.

Al separar por tipo de pozo, observamos que Random Forest mejora las métricas en los grupos mayoritarios de pozos petrolíferos y gasíferos. En cambio, presenta errores mayores que la persistencia en algunos grupos con pocas observaciones y producción real igual a cero. Estos grupos pequeños deben interpretarse con cautela, ya que unos pocos errores pueden modificar considerablemente sus métricas.

Por lo tanto, el resultado general es favorable, pero el desempeño no es igual para todos los tipos de pozo. Este diagnóstico orienta los próximos experimentos; no se utilizó el conjunto de test ni se modificó el modelo.

### 14. Estrategia de modelado en dos etapas

En los resultados anteriores observamos que una parte importante de los registros tiene producción de petróleo igual a cero. También vimos que el modelo de regresión puede estimar una producción positiva para algunos casos cuya producción real termina siendo cero.

Para analizar este comportamiento, probaremos una estrategia de modelado en dos etapas. Primero, un clasificador estimará si la producción de petróleo del mes siguiente será cero o positiva. Luego, un modelo de regresión estimará la cantidad producida para los casos con producción positiva.

Las dos etapas se combinan para obtener una predicción final en metros cúbicos. Evaluaremos esa predicción en el conjunto de validación y la compararemos con la referencia de persistencia y con el Random Forest de regresión que ya probamos. El conjunto de test se mantiene apartado.

Esta estrategia no garantiza por sí sola un mejor resultado. La probaremos para comprobar si distinguir primero entre producción nula y positiva ayuda a reducir los errores, en particular las predicciones positivas para casos cuya producción real es cero.

#### 14.1. Primer componente: clasificación de producción positiva

Comenzamos creando una variable binaria a partir de `prod_pet_mes_siguiente`: toma el valor 1 cuando la producción del mes siguiente es positiva y 0 cuando es igual a cero.

Entrenamos un clasificador Random Forest con todos los registros de entrenamiento, ya que necesitamos ejemplos de ambos casos. El preprocesamiento se ajusta solo con entrenamiento y luego se aplica a validación. En esta etapa revisamos la matriz de confusión y métricas de clasificación para conocer la capacidad del modelo de distinguir las dos clases.

Además, guardamos la probabilidad estimada de producción positiva. Más adelante la utilizaremos, junto con las estimaciones del modelo de regresión, para construir y comparar distintas formas de obtener la predicción final en metros cúbicos.

Las métricas de clasificación son diagnósticas: el objetivo principal del proyecto sigue siendo estimar la cantidad de petróleo producida el mes siguiente.

In [15]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline

# Creamos una etiqueta binaria: 1 si la producción futura es positiva, 0 si es cero.
y_train_positivo = (y_train > 0).astype(int)
y_validation_positivo = (y_validation > 0).astype(int)

print("Distribución de las clases en entrenamiento:")
display(
    y_train_positivo
    .value_counts()
    .rename(index={0: "Producción igual a cero", 1: "Producción positiva"})
    .to_frame("Registros")
)

# Reutilizamos el preprocesamiento del Random Forest de regresión.
# El pipeline lo ajustará de nuevo usando únicamente el entrenamiento.
clasificador_produccion = Pipeline([
    (
        "preprocesamiento",
        clone(modelo_regresion.named_steps["preprocesamiento"])
    ),
    (
        "modelo",
        RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            max_features=0.7,
            random_state=42,
            n_jobs=-1
        )
    )
])

# El clasificador aprende a distinguir producción cero de producción positiva.
clasificador_produccion.fit(X_train, y_train_positivo)

# En validación obtenemos tanto la clase estimada como
# la probabilidad de producción positiva.
clase_predicha = clasificador_produccion.predict(X_validation)
probabilidad_positiva = clasificador_produccion.predict_proba(X_validation)[:, 1]

# Resumimos la clasificación usando el umbral predeterminado de 0.5.
# Más adelante evaluaremos cómo combinar las probabilidades con la regresión.
matriz_confusion = confusion_matrix(
    y_validation_positivo,
    clase_predicha,
    labels=[0, 1]
)

display(
    pd.DataFrame(
        matriz_confusion,
        index=["Real: cero", "Real: positiva"],
        columns=["Predicho: cero", "Predicho: positiva"]
    )
)

print(f"Balanced accuracy: {balanced_accuracy_score(y_validation_positivo, clase_predicha):.3f}")
print(f"Precisión para producción positiva: {precision_score(y_validation_positivo, clase_predicha, zero_division=0):.3f}")
print(f"Recall para producción positiva: {recall_score(y_validation_positivo, clase_predicha, zero_division=0):.3f}")
print(f"F1 para producción positiva: {f1_score(y_validation_positivo, clase_predicha, zero_division=0):.3f}")
print(f"Probabilidad positiva promedio estimada: {probabilidad_positiva.mean():.3f}")

Distribución de las clases en entrenamiento:


,Registros
prod_pet_mes_siguiente,
Producción positiva,33601
Producción igual a cero,15851


,Predicho: cero,Predicho: positiva
Real: cero,4157,158
Real: positiva,154,9851


Balanced accuracy: 0.974
Precisión para producción positiva: 0.984
Recall para producción positiva: 0.985
F1 para producción positiva: 0.984
Probabilidad positiva promedio estimada: 0.692


#### 14.2. Segundo componente: regresión para los casos positivos

El clasificador estima si la producción de petróleo del mes siguiente será cero o positiva. El segundo componente debe estimar la cantidad producida cuando el objetivo es positivo.

Para entrenarlo utilizamos únicamente las filas de entrenamiento con `prod_pet_mes_siguiente` mayor que cero. De esta manera, el regresor aprende la cantidad de producción dentro de los casos positivos, sin intentar modelar también los ceros.

En validación combinamos las salidas de ambos componentes de dos maneras. En la primera, usamos la clase predicha para decidir entre cero y la cantidad estimada por el regresor. En la segunda, ponderamos la cantidad estimada por la probabilidad de producción positiva.

Comparamos ambas estrategias con la persistencia y los modelos de regresión ya evaluados, utilizando las mismas observaciones de validación. El conjunto de test no se utiliza en esta comparación.

In [16]:
from sklearn.base import clone

# Seleccionamos los ejemplos de entrenamiento con producción futura positiva.
mascara_positiva_train = y_train > 0

X_train_positivo = X_train.loc[mascara_positiva_train]
y_train_positivo_cantidad = y_train.loc[mascara_positiva_train]

if X_train_positivo.empty:
    raise ValueError("No hay ejemplos positivos para entrenar el regresor.")

print(f"Ejemplos positivos para entrenar el regresor: {len(X_train_positivo):,}")
print(
    "Producción mínima del grupo positivo: "
    f"{y_train_positivo_cantidad.min():.2f} m³"
)

# Reutilizamos el Random Forest candidato, pero lo ajustamos
# únicamente con los ejemplos de producción positiva.
regresor_positivo = clone(modelo_rf_candidato)
regresor_positivo.fit(X_train_positivo, y_train_positivo_cantidad)

# Estimamos la cantidad para todos los registros de validación.
# La combinación con el clasificador decidirá cómo usar estas cantidades.
y_pred_cantidad_positiva = regresor_positivo.predict(X_validation)

# Estrategia 1: usamos la clase predicha (umbral predeterminado 0.5).
y_pred_dos_etapas_clase = np.where(
    clase_predicha == 1,
    y_pred_cantidad_positiva,
    0
)

# Estrategia 2: ponderamos la cantidad por la probabilidad de producción positiva.
y_pred_dos_etapas_probabilidad = (
    probabilidad_positiva * y_pred_cantidad_positiva
)

# Comparamos en las mismas filas de validación.
predicciones_dos_etapas = {
    "Persistencia": y_pred_referencia,
    "Random Forest regresión": y_pred_modelo,
    "Dos etapas: decisión por clase": y_pred_dos_etapas_clase,
    "Dos etapas: ponderada por probabilidad": y_pred_dos_etapas_probabilidad,
}

resultados_dos_etapas = pd.DataFrame({
    "MAE (m³)": {
        nombre: mean_absolute_error(y_validation, prediccion)
        for nombre, prediccion in predicciones_dos_etapas.items()
    },
    "RMSE (m³)": {
        nombre: np.sqrt(mean_squared_error(y_validation, prediccion))
        for nombre, prediccion in predicciones_dos_etapas.items()
    },
    "R²": {
        nombre: r2_score(y_validation, prediccion)
        for nombre, prediccion in predicciones_dos_etapas.items()
    },
    "Predicciones negativas": {
        nombre: (prediccion < 0).sum()
        for nombre, prediccion in predicciones_dos_etapas.items()
    },
})

display(resultados_dos_etapas.round(3))

Ejemplos positivos para entrenar el regresor: 33,601
Producción mínima del grupo positivo: 0.00 m³


,MAE (m³),RMSE (m³),R²,Predicciones negativas
Persistencia,160.039,506.071,0.850,0
Random Forest regresión,128.398,368.437,0.920,0
Dos etapas: decisión por clase,126.982,367.580,0.921,0
Dos etapas: ponderada por probabilidad,128.066,366.053,0.921,0
